# Bài 29: Cấu trúc từ chuyển động

Bài này lắp ráp so khớp đặc trưng (Bài 20), ước lượng bền vững (Bài 23), hiệu chỉnh (Bài 27), cùng ma trận thiết yếu và khôi phục tư thế (Bài 28) thành một quy trình hoàn chỉnh nhận các cặp tương ứng ảnh 2D từ hai góc nhìn và khôi phục **cả** chuyển động tương đối của các camera **lẫn** vị trí 3D của các điểm đang được quan sát &mdash; **C**ấu trúc **t**ừ **C**huyển động (**S**tructure **f**rom **M**otion — SfM). Thành phần duy nhất mà ta chưa phát triển chi tiết là **phép tam giác đạc (triangulation)**: biến một cặp điểm 2D đã khớp, cùng với các tư thế camera đã biết, thành một điểm 3D. (Bài 22 đã chỉ ra cách chuyển độ chênh lệch thành độ sâu trong trường hợp ảnh đã nắn thẳng; bài này tổng quát hóa ý tưởng đó.)

In [ ]:
# Download the sample images used in this lesson, if not already present
import os, urllib.request

for _f in ['bronzebull00.jpg', 'bronzebull02.jpg']:
    _path = f"../img/{_f}"
    if not os.path.exists(_path):
        os.makedirs("../img", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/sbirchfield/sbirchfield.github.io/main/cvintro/img/{_f}", _path)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

## Quy trình kinh điển, nhìn tổng quát

1. **So khớp đặc trưng** giữa hai ảnh (Bài 20: SIFT + kiểm tra tỉ số).
2. **Ước lượng ma trận thiết yếu** $E$ từ các cặp tương ứng đó, một cách bền vững (Bài 23: RANSAC), dùng tham số nội $K$ đã biết (Bài 27: hiệu chỉnh).
3. **Khôi phục tư thế tương đối** $(R, t)$ từ $E$ (Bài 28: `cv2.recoverPose`) &mdash; sai khác một hệ số tỉ lệ chưa biết trên $t$.
4. **Tam giác đạc**: với mỗi cặp đã khớp, giao hai tia tương ứng trong 3D để khôi phục một điểm 3D &mdash; được phát triển chi tiết lần đầu tiên trong bài này.

Ta dùng hai bức ảnh tổng hợp chân thực về một bức tượng đồng từ bộ dữ liệu BlendedMVS (<a href="../references.html#yao-2020-blendedmvs">Yao et al., 2020</a>). Khác với một cặp ảnh tự chụp, các ảnh này đi kèm với tư thế camera, thông số hiệu chỉnh và hình học chuẩn để kiểm tra toàn bộ quy trình.

## Hai bức ảnh thật, với tư thế camera đã biết

Như vừa nêu, mỗi ảnh đi kèm một bộ $(K, R, t)$ chuẩn. Ta coi camera của bức ảnh đầu tiên là gốc tọa độ thế giới và biểu diễn tư thế của camera thứ hai tương đối với nó &mdash; cho ta `P1` và `P2_true`, hai ma trận chiếu chuẩn mà cả bài này cố gắng khôi phục chỉ từ các cặp tương ứng ảnh.

In [ ]:
K = np.array([[583.2225, 0., 255.4034],
              [0., 583.2225, 188.8833],
              [0., 0., 1.]])

R1, t1 = np.eye(3), np.zeros(3)
R_true = np.array([[ 0.92083152, -0.14714329,  0.36113446],
                    [ 0.12698365,  0.98874714,  0.07907683],
                    [-0.36870643, -0.02695794,  0.92915445]])
t_true = np.array([-0.78751177, 0.00981936, 0.03364542])

P1 = K @ np.hstack([R1, t1.reshape(3, 1)])
P2_true = K @ np.hstack([R_true, t_true.reshape(3, 1)])

view1 = cv2.imread('../img/bronzebull00.jpg')
view2 = cv2.imread('../img/bronzebull02.jpg')

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].imshow(cv2.cvtColor(view1, cv2.COLOR_BGR2RGB))
axes[0].set_title('View 1')
axes[1].imshow(cv2.cvtColor(view2, cv2.COLOR_BGR2RGB))
axes[1].set_title('View 2')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

<p class="image-attribution">Nguồn ảnh: <a href="https://github.com/YoYo000/BlendedMVS" target="_blank" rel="noopener">BlendedMVS</a> (CC BY 4.0)</p>

## Bước 1: so khớp đặc trưng

SIFT + kiểm tra tỉ số (Bài 20), chạy trực tiếp trên hai bức ảnh.

In [ ]:
gray1 = cv2.cvtColor(view1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(view2, cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()
kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)

bf = cv2.BFMatcher()
raw_matches = bf.knnMatch(des1, des2, k=2)
good_matches = [m for m, n in raw_matches if m.distance < 0.75 * n.distance]

x1 = np.float32([kp1[m.queryIdx].pt for m in good_matches])
x2 = np.float32([kp2[m.trainIdx].pt for m in good_matches])

print(f'keypoints: {len(kp1)} (view 1), {len(kp2)} (view 2)')
print(f'good matches after ratio test: {len(good_matches)}')

match_vis = cv2.drawMatches(view1, kp1, view2, kp2, good_matches, None,
                             flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
plt.figure(figsize=(10, 4))
plt.imshow(cv2.cvtColor(match_vis, cv2.COLOR_BGR2RGB))
plt.title('Feature matches')
plt.axis('off')
plt.show()

## Bước 2: ma trận thiết yếu bền vững và khôi phục tư thế

Đưa các điểm đã khớp qua cùng quy trình bền vững đã xây dựng ở Bài 23 và 27 để khôi phục ma trận thiết yếu và tư thế tương đối của các camera.

In [ ]:
E, inlier_mask = cv2.findEssentialMat(x1, x2, K, method=cv2.RANSAC, threshold=1.0)
_, R_estimated, t_estimated, _ = cv2.recoverPose(E, x1, x2, K)

print(f'inliers: {int(inlier_mask.sum())} / {len(inlier_mask)}')
print('recovered rotation:\n', np.round(R_estimated, 4))
print('true rotation:\n', np.round(R_true, 4))
print()
print('recovered translation direction:', np.round(t_estimated.ravel(), 4))
print('true translation direction:     ', np.round(t_true / np.linalg.norm(t_true), 4))

Như ở Bài 28, `recoverPose` trả về một hướng tịnh tiến có **độ dài đơn vị** &mdash; khoảng cách đường cơ sở thực tế giữa các camera về bản chất không thể khôi phục chỉ từ các cặp tương ứng ảnh. Ta sẽ quay lại điểm này.

## Bước 3: tam giác đạc

Cho hai ma trận chiếu camera $P_1, P_2$ (mỗi ma trận $3\times4$, ánh xạ một điểm 3D tới một điểm ảnh 2D trong tọa độ thuần nhất) và một cặp điểm ảnh đã khớp $(x_1, x_2)$, ta muốn tìm điểm 3D $X$ thỏa mãn cả $x_1 \propto P_1 X$ lẫn $x_2 \propto P_2 X$. Mỗi góc nhìn đóng góp 2 phương trình tuyến tính độc lập theo 4 ẩn thuần nhất của $X$ (nhân chéo để khử hệ số tỉ lệ chưa biết), theo cùng công thức DLT/SVD đã dùng cho phép đồng nhất (Bài 24) và ma trận cơ bản (Bài 28):

$$A = \begin{bmatrix} u_1 P_1^{(3)} - P_1^{(1)} \\ v_1 P_1^{(3)} - P_1^{(2)} \\ u_2 P_2^{(3)} - P_2^{(1)} \\ v_2 P_2^{(3)} - P_2^{(2)} \end{bmatrix}, \qquad AX = 0$$

trong đó $P^{(i)}$ ký hiệu hàng $i$ của $P$. Không gian rỗng của $A$ (vector suy biến phải ứng với giá trị suy biến nhỏ nhất) cho ra $X$.

In [ ]:
def triangulate_dlt(P1, P2, x1, x2):
    points = []
    for (u1, v1), (u2, v2) in zip(x1, x2):
        A = np.array([
            u1 * P1[2] - P1[0],
            v1 * P1[2] - P1[1],
            u2 * P2[2] - P2[0],
            v2 * P2[2] - P2[1],
        ])
        _, _, Vt = np.linalg.svd(A)
        X = Vt[-1]
        points.append(X[:3] / X[3])
    return np.array(points)

# validate against the dataset's TRUE camera poses: triangulate the RANSAC inliers with them,
# and check that reprojecting the resulting 3D points lands back on the original pixels
inliers = inlier_mask.ravel().astype(bool)
x1_in, x2_in = x1[inliers], x2[inliers]

recon_true_cameras = triangulate_dlt(P1, P2_true, x1_in, x2_in)

cv_points_4d = cv2.triangulatePoints(P1, P2_true, x1_in.T, x2_in.T)
cv_points_3d = (cv_points_4d[:3] / cv_points_4d[3]).T
print(f'max diff vs. cv2.triangulatePoints: {np.abs(recon_true_cameras - cv_points_3d).max():.2e}')

# cheirality check: a correctly-matched 3D point must be in front of BOTH cameras --
# a real pipeline always applies this, since epipolar-only RANSAC can still admit a bad match
depth1 = recon_true_cameras[:, 2]
depth2 = (R_true @ recon_true_cameras.T + t_true.reshape(3, 1))[2]
valid = (depth1 > 0) & (depth2 > 0)
x1_in, x2_in, recon_true_cameras = x1_in[valid], x2_in[valid], recon_true_cameras[valid]
print(f'kept {valid.sum()} / {len(valid)} inliers after cheirality check')

def reprojection_error(P, X, x):
    X_h = np.hstack([X, np.ones((len(X), 1))])
    proj = (P @ X_h.T).T
    proj = proj[:, :2] / proj[:, 2:3]
    return np.linalg.norm(proj - x, axis=1)

err1 = reprojection_error(P1, recon_true_cameras, x1_in)
err2 = reprojection_error(P2_true, recon_true_cameras, x2_in)
all_err = np.concatenate([err1, err2])
print(f'reprojection error using TRUE camera poses: mean {all_err.mean():.3f} px, max {all_err.max():.3f} px')

### Tính chirality: một phép kiểm tra miễn phí

Phân rã $E$ thành $(R, t)$ ở Bước 2 thực ra cho ra bốn tổ hợp dấu hợp lệ về mặt toán học; chỉ một tổ hợp đặt các điểm quan sát được ở phía trước *cả hai* camera, và `cv2.recoverPose` dùng chính sự thật đó &mdash; gọi là **ràng buộc chirality (cheirality constraint)** &mdash; để chọn đúng tổ hợp bên trong, bằng cách tam giác đạc vài điểm và kiểm tra độ sâu của chúng. Phép kiểm tra ở trên áp dụng tường minh cùng ý tưởng đó: riêng ràng buộc epipolar không loại trừ được một cặp tương ứng tam giác đạc ra điểm nằm *phía sau* camera, nên một điểm khớp nhầm vẫn có thể vượt qua phép kiểm tra điểm trong của RANSAC. Yêu cầu độ sâu dương ở cả hai góc nhìn là một bộ lọc bổ sung miễn phí bên trên đó &mdash; chính là thứ đã loại bỏ một điểm trong bị đánh dấu ở trên.

## Ghép lại: tái tạo bằng tư thế *ước lượng được*

Giờ là phép thử thực sự: tam giác đạc bằng ma trận camera dựng từ $(R, t)$ *ước lượng được* của `recoverPose`, không phải tư thế thực của bộ dữ liệu. Vì $t$ chỉ được khôi phục sai khác một hệ số tỉ lệ, phép tái tạo cũng vậy &mdash; mọi điểm 3D đều nhỏ hơn thực tế một hệ số cố định. Nhân với độ dài đường cơ sở thực (một chút gian lận, chỉ để đánh giá) sẽ khôi phục được cảnh theo đúng đơn vị đo thực.

In [ ]:
P2_estimated = K @ np.hstack([R_estimated, t_estimated.reshape(3, 1)])
reconstruction_unit_scale = triangulate_dlt(P1, P2_estimated, x1_in, x2_in)

true_baseline = np.linalg.norm(t_true)
reconstruction_metric = reconstruction_unit_scale * true_baseline

error = np.linalg.norm(reconstruction_metric - recon_true_cameras, axis=1)
print(f'true baseline: {true_baseline:.3f}')
print(f'mean 3D reconstruction error (vs. triangulation from true poses): {error.mean():.3f}')
print(f'max 3D reconstruction error:                                     {error.max():.3f}')

Với các cặp tương ứng thật (không hoàn hảo), toàn bộ quy trình &mdash; ma trận thiết yếu, tư thế, tam giác đạc &mdash; tái tạo cảnh với sai số chỉ bằng một phần nhỏ của đường cơ sở camera, chỉ dùng các cặp tương ứng điểm ảnh 2D và tham số nội đã biết, cộng thêm *một* con số bên ngoài (đường cơ sở thực) để khắc phục sự mơ hồ về tỉ lệ. Trên thực tế, tham chiếu tỉ lệ đó có thể đến từ kích thước đã biết của một vật trong cảnh, một cảm biến thứ hai (GPS, IMU, LiDAR), hoặc một hệ ảnh nổi đã hiệu chỉnh (Bài 22) thay vì hai camera độc lập tùy ý.

### Trực quan hóa phép tái tạo

Để có một đám mây điểm dày đặc hơn, dễ nhận ra hơn so với tập điểm trong RANSAC nghiêm ngặt dùng cho phép kiểm tra số ở trên, ta nới lỏng các ngưỡng so khớp đặc trưng (`SIFT_create(contrastThreshold=0.01)`, tỉ số 0.85) để lấy thêm nhiều cặp tương ứng hơn &mdash; nhiễu hơn một chút &mdash; rồi tô màu mỗi điểm 3D bằng màu điểm ảnh thực của nó từ góc nhìn 1.

In [ ]:
sift_dense = cv2.SIFT_create(contrastThreshold=0.01)
kp1d, des1d = sift_dense.detectAndCompute(gray1, None)
kp2d, des2d = sift_dense.detectAndCompute(gray2, None)
raw_dense = bf.knnMatch(des1d, des2d, k=2)
good_dense = [m for m, n in raw_dense if m.distance < 0.85 * n.distance]

x1d = np.float32([kp1d[m.queryIdx].pt for m in good_dense])
x2d = np.float32([kp2d[m.trainIdx].pt for m in good_dense])

_, dense_mask = cv2.findEssentialMat(x1d, x2d, K, method=cv2.RANSAC, threshold=1.0)
dense_inliers = dense_mask.ravel().astype(bool)
x1d, x2d = x1d[dense_inliers], x2d[dense_inliers]

recon_dense_true = triangulate_dlt(P1, P2_true, x1d, x2d)
recon_dense_estimated = triangulate_dlt(P1, P2_estimated, x1d, x2d) * true_baseline

# cheirality check again, same as for the sparse set above
depth1 = recon_dense_true[:, 2]
depth2 = (R_true @ recon_dense_true.T + t_true.reshape(3, 1))[2]
valid = (depth1 > 0) & (depth2 > 0)
x1d, recon_dense_true, recon_dense_estimated = x1d[valid], recon_dense_true[valid], recon_dense_estimated[valid]

rgb1 = cv2.cvtColor(view1, cv2.COLOR_BGR2RGB)
px = np.clip(x1d[:, 0].astype(int), 0, rgb1.shape[1] - 1)
py = np.clip(x1d[:, 1].astype(int), 0, rgb1.shape[0] - 1)
point_colors = rgb1[py, px] / 255.0

print(f'{len(x1d)} points in the denser cloud (vs. {len(x1_in)} used for the numeric check above)')

In [ ]:
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(*recon_dense_true.T, c='gray', s=8, alpha=0.3, label='triangulated with true poses')
ax.scatter(*recon_dense_estimated.T, c=point_colors, s=25, label='reconstructed (estimated pose)')

# mark the two camera centers
cam1_center = -R1.T @ t1
cam2_center = -R_true.T @ t_true
ax.scatter(*cam1_center, c='blue', s=80, marker='^', label='camera 1')
ax.scatter(*cam2_center, c='green', s=80, marker='^', label='camera 2')

ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.legend(fontsize=8)
ax.set_title('Structure from motion: recovered 3D points and camera poses')
plt.show()

### Từ thưa đến dày đặc

Ngay cả với các ngưỡng đã nới lỏng ở trên, so khớp đặc trưng chỉ cho ra một đám mây *thưa* &mdash; một điểm 3D cho mỗi điểm đặc trưng nổi bật trong ảnh. Như đã thấy ở trên, đám mây thưa thường không thỏa mãn lắm. Nhưng khi tư thế camera đã biết, ta không còn bị giới hạn ở các điểm đặc trưng nổi bật: **ảnh nổi đa góc nhìn (multi-view stereo — MVS)** ước lượng độ sâu cho *mọi* điểm ảnh trong *mọi* góc nhìn, bằng cách quét một giả thuyết mặt phẳng hoặc một mảng (patch) qua không gian (một sự tổng quát hóa so khớp ảnh nổi ở Bài 22 từ một cặp đã nắn thẳng sang các camera đã hiệu chỉnh có tư thế tùy ý). Các bản đồ độ sâu theo từng góc nhìn đó sau đó được hợp nhất &mdash; lọc bỏ các ước lượng không nhất quán, gộp phần còn lại &mdash; thành đám mây điểm (hoặc lưới — mesh) có màu dày đặc mà các bộ công cụ như COLMAP (<a href="../references.html#schonberger-2016-colmap">Schönberger &amp; Frahm, 2016</a>) tạo ra.

### Từ hai góc nhìn đến nhiều góc nhìn: SfM tăng dần và tinh chỉnh bó

Các phép tái tạo thực tế hiếm khi dừng ở hai góc nhìn. Công thức chuẩn tuần tự đưa thêm các ảnh mới vào qua 3 bước: 1) so khớp đặc trưng giữa ảnh mới và một hoặc nhiều ảnh đã có; 2) tính tư thế camera mới tương đối với đám mây điểm 3D hiện có; và 3) cập nhật đám mây điểm 3D với các cặp khớp mới bằng tam giác đạc. Bước 2 đòi hỏi giải bài toán Perspective-n-Point (tức là, cho các cặp tương ứng 2D-3D và $K$, khôi phục tư thế của camera đó qua `cv2.solvePnP`).

Khi các góc nhìn được tích lũy, sai số cộng dồn &mdash; tư thế làm hỏng các điểm được tam giác đạc, rồi các điểm đó lại làm hỏng tư thế khôi phục tiếp theo, cứ thế tiếp diễn. **Tinh chỉnh bó (bundle adjustment)** khắc phục điều này bằng cách tinh chỉnh tất cả tư thế camera và điểm 3D *cùng nhau* trong một bài toán tối ưu bình phương tối thiểu phi tuyến lớn duy nhất, cực tiểu hóa tổng sai số chiếu lại cộng trên mọi điểm trong mọi góc nhìn cùng lúc. Tinh chỉnh bó có lẽ là bước quan trọng nhất phân biệt một quy trình hai góc nhìn đơn giản như trong bài này với một hệ thống SfM thực thụ (COLMAP, v.v.).

### Bài tập

1. Thêm nhiễu điểm ảnh tổng hợp (ví dụ `rng.normal(0, 1.0, x1_in.shape)`) bên trên các phát hiện vốn đã thật trong `x1_in`/`x2_in` trước khi tam giác đạc. Sai số tái tạo tăng bao nhiêu, và nó tăng đều, hay tệ hơn với các điểm xa camera hơn (mối quan hệ độ chênh lệch-độ sâu ở Bài 22: điểm xa tạo ra thị sai nhỏ hơn, nhiễu hơn)?
2. Tra cứu chữ ký hàm của `cv2.solvePnP` và phác thảo (bằng lời) cách bạn mở rộng notebook này cho góc nhìn thứ ba: bạn sẽ so khớp những điểm 3D nào từ phép tái tạo `view1`/`view2` vào ảnh mới, và đầu vào/đầu ra của `solvePnP` sẽ ánh xạ thế nào lên các thành phần ta đã có (`K`, các điểm 2D đã khớp, và đám mây điểm 3D hiện có)?
3. Cặp này (góc nhìn 0 và 2) có đường cơ sở được chọn rộng rãi có chủ đích để so khớp đáng tin cậy. Thay vào đó hãy dùng góc nhìn 0 và 1 (đường cơ sở ngắn hơn nhiều &mdash; bạn sẽ cần các tham số camera `.npz` của chúng từ cùng cảnh BlendedMVS). Đường cơ sở ngắn hơn ảnh hưởng thế nào đến số điểm trong RANSAC, và độ chính xác của tư thế và phép tái tạo khôi phục được, so với cặp có đường cơ sở rộng hơn dùng ở trên?